In [ ]:
## Preprocessing & Feature Extraction

In [ ]:
"""
STEP 1: EEG Data Preprocessing, Feature Extraction,
        and Optional Artifact Removal with Wiener Filter
        and Pre-selection (VarianceThreshold + optional PCA).
"""

import os
import numpy as np
import pandas as pd

import mne
from mne.decoding import CSP
from scipy.signal import wiener, welch
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

# ---------------------------------------------------------------------
# 1. HELPER FUNCTIONS (Preprocessing + Feature Extraction)
# ---------------------------------------------------------------------

def find_eeg_folder(subject_folder_path):
    """
    Recursively searches for an 'eeg' subfolder within the given subject folder path.
    Returns the full path if found, else raises FileNotFoundError.
    """
    for root, dirs, files in os.walk(subject_folder_path):
        if 'eeg' in dirs:
            return os.path.join(root, 'eeg')
    raise FileNotFoundError(f"EEG folder not found in {subject_folder_path}")


def load_and_preprocess_data(condition_type, subject_folder, directory_path):
    """
    Loads the .bdf EEG file for a given subject, applies band-pass filtering (0.5 - 40 Hz),
    reads the corresponding events file (_events.tsv), and epochs data (-0.2s to 2.0s).
    Returns:
        eeg_data_epochs (numpy array): shape (n_epochs, n_channels, n_times).
    """
    subject_folder_path = os.path.join(directory_path, subject_folder)
    eeg_folder_path = find_eeg_folder(subject_folder_path)
    
    # Find .bdf file
    bdf_files = [f for f in os.listdir(eeg_folder_path) if f.endswith('.bdf')]
    if not bdf_files:
        raise FileNotFoundError(f"No .bdf file found in {eeg_folder_path}")
    bdf_path = os.path.join(eeg_folder_path, bdf_files[0])

    # Load raw BDF
    raw_data = mne.io.read_raw_bdf(bdf_path, preload=True)
    
    # Band-pass filter: 0.5 - 40 Hz
    raw_data_clean = raw_data.copy().filter(l_freq=0.5, h_freq=40.0)

    # Find events file
    event_files = [f for f in os.listdir(eeg_folder_path) if f.endswith('_events.tsv')]
    if not event_files:
        raise FileNotFoundError(f"No events.tsv file found in {eeg_folder_path}")
    events_df = pd.read_csv(os.path.join(eeg_folder_path, event_files[0]), sep='\t')
    
    # Build MNE events array
    sfreq = raw_data_clean.info['sfreq']
    events_mne = np.zeros((len(events_df), 3), int)
    events_mne[:, 0] = (events_df['onset'] * sfreq).astype(int)  # Convert onset (seconds) to samples
    events_mne[:, 2] = events_df['value']  # Use 'value' as the event ID

    # Create epochs from -0.2s to 2.0s
    epochs = mne.Epochs(raw_data_clean, events_mne, event_id=None,
                        tmin=-0.2, tmax=2.0, preload=True)

    # Return the epoch data array
    return epochs.get_data()  # shape: (n_epochs, n_channels, n_times)


def extract_csp_features(eeg_data, labels):
    """
    Extract CSP features (4 components, log-variance).
    Returns (n_epochs, 4).
    """
    csp = CSP(n_components=4, reg=None, log=True, cov_est='epoch')
    csp.fit(eeg_data, labels)
    return csp.transform(eeg_data)


def extract_swlngp_features(eeg_data):
    """
    SWLNGP: threshold-based binary encoding for each channel.
    Returns (n_epochs, n_channels).
    """
    swlngp_features = []
    for trial in eeg_data:  # (n_channels, n_times)
        trial_feats = []
        for channel in trial:
            mean_chan = np.mean(channel)
            threshold = np.mean(np.abs(channel - mean_chan))
            binary_pattern = np.abs(channel - mean_chan) > threshold
            swlngp_val = np.sum(binary_pattern.astype(int))
            trial_feats.append(swlngp_val)
        swlngp_features.append(trial_feats)
    return np.array(swlngp_features)


def extract_spv_features(eeg_data, sfreq=256):
    """
    SPV: Mean Welch PSD across all frequencies per channel.
    Returns (n_epochs, n_channels).
    """
    spv_feats = []
    for trial in eeg_data:
        trial_arr = []
        for channel in trial:
            f, Pxx = welch(channel, fs=sfreq, nperseg=256)
            trial_arr.append(np.mean(Pxx))
        spv_feats.append(trial_arr)
    return np.array(spv_feats)


# ---------------------------------------------------------------------
# 2. MAIN ROUTINE: LOAD DATA, PREPROCESS, FEATURE EXTRACTION
#    + Optional Wiener Filter + Feature Reduction
# ---------------------------------------------------------------------

if __name__ == "__main__":

    # ========== Paths and Subject Lists ==========
    directory_path = r"K:\Academic content\Paper 3\Dataset\Dataset1 (UC San diego)\ds002778"

    hc_subjects = [
        'sub-hc1','sub-hc2','sub-hc4','sub-hc7','sub-hc8',
        'sub-hc10','sub-hc18','sub-hc20','sub-hc21','sub-hc24',
        'sub-hc25','sub-hc29','sub-hc31','sub-hc31','sub-hc32','sub-hc33'
    ]
    pd_subjects = [
        'sub-pd17','sub-pd3','sub-pd5','sub-pd6','sub-pd9',
        'sub-pd11','sub-pd12','sub-pd13','sub-pd14','sub-pd16',
        'sub-pd19','sub-pd22','sub-pd23','sub-pd26','sub-pd28'
    ]

    # ========== 2.1 Load & Preprocess Data (HC, PD_ON, PD_OFF) ==========
    # Healthy Controls
    eeg_data_hc = np.vstack([
        load_and_preprocess_data('HC', subj, directory_path) 
        for subj in hc_subjects
    ])
    # PD_ON
    eeg_data_pd_on = np.vstack([
        load_and_preprocess_data('PD_ON', subj, directory_path)
        for subj in pd_subjects
    ])
    # PD_OFF
    eeg_data_pd_off = np.vstack([
        load_and_preprocess_data('PD_OFF', subj, directory_path)
        for subj in pd_subjects
    ])

    # --------------------------------------------------------
    # Optional: Wiener Filter for artifact removal
    # Example: apply to HC data
    from scipy.signal import wiener

    # If you want to apply Wiener filter to HC data
    eeg_data_hc = np.array([
        wiener(epoch) for epoch in eeg_data_hc
    ])

    # Similarly, you could do:
    eeg_data_pd_on = np.array([wiener(epoch) for epoch in eeg_data_pd_on])
    eeg_data_pd_off = np.array([wiener(epoch) for epoch in eeg_data_pd_off])
    #
    # ... if you want to apply Wiener filtering to PD data as well.
    # --------------------------------------------------------

    # ========== 2.2 Combine Data & Labels ==========
    eeg_data_combined = np.vstack([eeg_data_hc, eeg_data_pd_on, eeg_data_pd_off])

    labels_hc = np.zeros(len(eeg_data_hc), dtype=int)  # 0 = HC
    labels_pd = np.ones(len(eeg_data_pd_on) + len(eeg_data_pd_off), dtype=int)  # 1 = PD

    labels_combined = np.hstack([labels_hc, labels_pd])

    # ========== 2.3 Feature Extraction ==========
    X_csp = extract_csp_features(eeg_data_combined, labels_combined)
    X_swlngp = extract_swlngp_features(eeg_data_combined)
    X_spv = extract_spv_features(eeg_data_combined)

    # Concatenate all extracted features
    X_combined = np.hstack([X_csp, X_swlngp, X_spv])
    print("Raw extracted features shape:", X_combined.shape)

    # ========== 2.5 Save to CSV ==========
    df_final = pd.DataFrame(X_combined)
    df_final['label'] = labels_combined
    df_final.to_csv('reduced_features_and_labels.csv', index=False)
    print("Step 1 complete. File 'reduced_features_and_labels.csv' created with final features + labels.")
